# ============================================================
# HYROX BILBAO 2026 — SCRAPING
# Fuente: results.hyrox.com
# División: HYROX DOUBLES MEN
# Evento: Bilbao 2026
# ============================================================

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import os

In [ ]:
def extraer_pagina(ruta_archivo, num_pagina):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    filas = soup.find_all('li', class_=lambda c: c and 'list-group-item' in c
                          and 'row' in c and 'list-group-header' not in c)

    registros = []
    for fila in filas:
        pos_general  = fila.find('div', class_=lambda c: c and 'place-primary' in c)
        pos_categoria = fila.find('div', class_=lambda c: c and 'place-secondary' in c)
        nombres      = fila.find('h4',  class_=lambda c: c and 'type-relay_member' in c)
        edad         = fila.find('div', class_=lambda c: c and 'type-age_class' in c)
        tiempo       = fila.find('div', class_=lambda c: c and 'type-time' in c)

        for tag in [edad, tiempo]:
            if tag:
                label = tag.find('div', class_='list-label')
                if label: label.decompose()

        registros.append({
            'pos_general':    pos_general.text.strip()  if pos_general  else None,
            'pos_categoria':  pos_categoria.text.strip() if pos_categoria else None,
            'nombres_pareja': nombres.text.strip()      if nombres      else None,
            'grupo_edad':     edad.text.strip()         if edad         else None,
            'tiempo_total':   tiempo.text.strip()       if tiempo       else None,
            'pagina':         num_pagina
        })

    return registros

In [ ]:
archivos_html = sorted([f for f in os.listdir('/content') if f.endswith('.html')])
print("Archivos HTML encontrados:")
for a in archivos_html:
    print(f"  {a}")

Archivos HTML encontrados:
  hyox_page0.html
  hyrox_page1.html
  hyrox_page2.html
  hyrox_page3.html
  hyrox_page4.html
  hyrox_page5.html
  hyrox_page6.html
  hyrox_page7.html
  hyrox_page8.html


In [ ]:
todos_los_datos = []

archivos = ['hyox_page0.html'] + [f'hyrox_page{i}.html' for i in range(1, 9)]

for num, nombre in enumerate(archivos):
    ruta = f'/content/{nombre}'
    datos = extraer_pagina(ruta, num)
    todos_los_datos.extend(datos)
    print(f'✅ {nombre}: {len(datos)} registros')

df_raw = pd.DataFrame(todos_los_datos)
print(f'\nTotal registros: {len(df_raw)}')

✅ hyox_page0.html: 100 registros
✅ hyrox_page1.html: 100 registros
✅ hyrox_page2.html: 100 registros
✅ hyrox_page3.html: 100 registros
✅ hyrox_page4.html: 100 registros
✅ hyrox_page5.html: 100 registros
✅ hyrox_page6.html: 100 registros
✅ hyrox_page7.html: 100 registros
✅ hyrox_page8.html: 61 registros

Total registros: 861


In [ ]:
df_raw['pos_general_num'] = pd.to_numeric(df_raw['pos_general'], errors='coerce')

print("=== SHAPE ===")
print(df_raw.shape)

print("\n=== TIPOS DE DATOS ===")
print(df_raw.dtypes)

print("\n=== NULOS POR COLUMNA ===")
print(df_raw.isnull().sum())

print(f"\n=== RANGO POSICIONES ===")
print(f"{df_raw['pos_general_num'].min()} → {df_raw['pos_general_num'].max()}")

print("\n=== DISTRIBUCIÓN GRUPO DE EDAD ===")
print(df_raw['grupo_edad'].value_counts().sort_index())

print("\n=== MUESTRA ===")
print(df_raw.drop(columns='pos_general_num').head(10).to_string())

df_raw = df_raw.drop(columns='pos_general_num')

=== SHAPE ===
(861, 7)

=== TIPOS DE DATOS ===
pos_general        object
pos_categoria      object
nombres_pareja     object
grupo_edad         object
tiempo_total       object
pagina              int64
pos_general_num     int64
dtype: object

=== NULOS POR COLUMNA ===
pos_general        0
pos_categoria      0
nombres_pareja     0
grupo_edad         0
tiempo_total       0
pagina             0
pos_general_num    0
dtype: int64

=== RANGO POSICIONES ===
1 → 855

=== DISTRIBUCIÓN GRUPO DE EDAD ===
grupo_edad
16-24     75
25-29    198
30-34    245
35-39    173
40-44    103
45-49     37
50-54     22
55-59      3
60-64      5
Name: count, dtype: int64

=== MUESTRA ===
  pos_general pos_categoria                                  nombres_pareja grupo_edad tiempo_total  pagina
0           1             1                 Álvaro Villegas, Teresa Bartret      25-29     01:07:44       0
1           1             1  JOSE AGUSTIN ALISES GIMENEZ, LUIS GARCIA RUBIO      25-29     00:50:14       0
2    

In [ ]:
df_raw.to_csv('hyrox_bilbao_2026_raw.csv', index=False)
print("✅ Guardado: hyrox_bilbao_2026_raw.csv")
print(f"   {len(df_raw)} registros | {df_raw.shape[1]} columnas")

✅ Guardado: hyrox_bilbao_2026_raw.csv
   861 registros | 6 columnas
